In [ ]:
!pip install --quiet datasets evaluate transformers[sentencepiece]

import pandas as pd
from datasets import Dataset, load_dataset
from transformers import GPT2Tokenizer, GPT2ForSequenceClassification
from torch.utils.data import Dataset as TorchDataset, DataLoader
import torch
import torch.nn as nn
import torch.nn.functional as F
import evaluate
import numpy as np

df = pd.read_parquet("hf://datasets/ucirvine/sms_spam/plain_text/train-00000-of-00001.parquet")
hf_dataset = Dataset.from_pandas(df)
train_ds = hf_dataset.select(range(4000))
val_ds = hf_dataset.select(range(4000, 5000))
df.head()

model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

def tokenize_fn(examples):
    return tokenizer(examples["sms"], padding="max_length", truncation=True, max_length=64)

train_tok = train_ds.map(tokenize_fn, batched=True)
val_tok = val_ds.map(tokenize_fn, batched=True)

model = GPT2ForSequenceClassification.from_pretrained(model_name, num_labels=2, pad_token_id=tokenizer.eos_token_id)

import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn

class Attention(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.scale = embed_dim ** -0.5
    def forward(self, query, key, value, mask=None):
        scores = torch.matmul(query, key.transpose(-2, -1)) * self.scale
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        return torch.matmul(attn, value), attn

class SimpleAttentionClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.attn = Attention(embed_dim)
        self.fc = nn.Linear(embed_dim, num_classes)
    def forward(self, x):
        embed = self.embedding(x)
        attn_output, _ = self.attn(embed, embed, embed)
        pooled = attn_output.mean(dim=1)
        return self.fc(pooled)

def preprocess_for_attention(example):
    tokens = tokenizer.encode(example["sms"], max_length=64, truncation=True, padding="max_length")
    return {"input_ids": tokens, "label": example["label"]}

train_ds_attn = train_ds.map(preprocess_for_attention)
val_ds_attn = val_ds.map(preprocess_for_attention)

class SMSDataset(TorchDataset):
    def __init__(self, hf_dataset):
        self.data = hf_dataset
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            'input_ids': torch.tensor(item["input_ids"], dtype=torch.long),
            'label': torch.tensor(item["label"], dtype=torch.long)
        }

train_loader = DataLoader(SMSDataset(train_ds_attn), batch_size=32, shuffle=True)
val_loader = DataLoader(SMSDataset(val_ds_attn), batch_size=32)

vocab_size = len(tokenizer)
embed_dim = 64
num_classes = 2
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
attn_model = SimpleAttentionClassifier(vocab_size, embed_dim, num_classes).to(device)
optimizer = torch.optim.Adam(attn_model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

attn_model.train()
for batch in train_loader:
    inputs = batch['input_ids'].to(device)
    labels = batch['label'].to(device)
    optimizer.zero_grad()
    outputs = attn_model(inputs)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()
print("Custom Attention model trained on SMS dataset. Sample batch loss:", loss.item())

accuracy = evaluate.load("accuracy")
precision = evaluate.load("precision")
recall = evaluate.load("recall")
f1 = evaluate.load("f1")

def compute_metrics(pred):
    logits, labels = pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "precision": precision.compute(predictions=preds, references=labels)["precision"],
        "recall": recall.compute(predictions=preds, references=labels)["recall"],
        "f1": f1.compute(predictions=preds, references=labels)["f1"]
    }

print("\n📊 Evaluating GPT-2 Model...")
gpt2_preds = []
gpt2_labels = []
model.eval()
for ex in val_tok:
    inputs = torch.tensor(ex['input_ids']).unsqueeze(0).to(model.device)
    with torch.no_grad():
        logits = model(inputs).logits
    pred = torch.argmax(logits, dim=-1).cpu().item()
    gpt2_preds.append(pred)
    gpt2_labels.append(ex['label'])
gpt2_metrics = {
    "accuracy":  accuracy.compute(predictions=gpt2_preds, references=gpt2_labels)["accuracy"],
    "precision": precision.compute(predictions=gpt2_preds, references=gpt2_labels)["precision"],
    "recall":    recall.compute(predictions=gpt2_preds, references=gpt2_labels)["recall"],
    "f1":        f1.compute(predictions=gpt2_preds, references=gpt2_labels)["f1"]
}
print("GPT-2 Metrics:", gpt2_metrics)

print("\n📊 Evaluating Custom Attention Model...")
attn_preds = []
attn_labels = []
attn_model.eval()
for batch in val_loader:
    inputs = batch['input_ids'].to(device)
    labels = batch['label'].to(device)
    with torch.no_grad():
        outputs = attn_model(inputs)
        preds = torch.argmax(outputs, dim=1)
    attn_preds.extend(preds.cpu().tolist())
    attn_labels.extend(labels.cpu().tolist())
attn_metrics = {
    "accuracy":  accuracy.compute(predictions=attn_preds, references=attn_labels)["accuracy"],
    "precision": precision.compute(predictions=attn_preds, references=attn_labels)["precision"],
    "recall":    recall.compute(predictions=attn_preds, references=attn_labels)["recall"],
    "f1":        f1.compute(predictions=attn_preds, references=attn_labels)["f1"]
}
print("Attention Model Metrics:", attn_metrics)

print("\n🧠 Reflection Answers:")
print("1. Query represents the current token being processed, keys represent all tokens providing context, values hold the information extracted from those tokens.")
print("2. The scaling factor prevents large dot products from causing softmax saturation, ensuring stable gradients.")
print("3. Self-attention processes sequences in parallel, captures long dependencies efficiently, and avoids sequential limitations of RNNs.")
print("4. GPT-2 performs better due to pre-training, while the custom model is simpler but trainable; improvements include deeper layers, dropout, and more data.")
